# Run diffusion inference / translation (EAF GPU)

**Kernel:** `conda env:.conda-diffusion`

Wraps `inference/run_ddim2ddim_inference.py` and `inference/run_mixed_diffusion_inference.py`.
Outputs go to scratch for speed; copy summaries to `/exp/sbnd/data/.../inference` when durable storage is needed.

Modes: `ddim2ddim` | `rand2ddim` | `rand2ddpm` | `ddpm2ddpm` | `ddim2ddpm`.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
sys.path.insert(0, str(APP_ROOT))
from configs.paths import (
    DATA_ROOT,
    SCRATCH_INFERENCE,
    default_model_checkpoint,
    ensure_layout,
    resolve_inference_dirs,
)

ensure_layout()

discover = {}
for p in [DATA_ROOT / "training" / "eaf_discover.json", APP_ROOT / "train" / "eaf_discover.json"]:
    if p.is_file():
        discover = json.loads(p.read_text())
        break

MODEL_PATH = default_model_checkpoint()
# Prefer anisotropic / other variants under archive/ICARUS_NNs when needed:
# MODEL_PATH = DATA_ROOT / "archive" / "ICARUS_NNs" / "anisotropic_best.pt"

MODE = "ddim2ddim"   # or rand2ddpm, etc.
# One or more T values (ddim2ddim supports a sweep; mixed modes use the first only for now)
T_LIST = [50, 100, 200, 400]
INPUT_DIR = Path("/scratch/7DayLifetime/munjung/ICARUS/plane1_healthy")  # edit
OUTPUT_DIR = SCRATCH_INFERENCE / f"plane1_healthy_outputs{'' if MODE=='ddim2ddim' else '-'+MODE}"

print("MODEL ", MODEL_PATH, MODEL_PATH.is_file())
print("INPUT ", INPUT_DIR, INPUT_DIR.exists())
print("OUTPUT", OUTPUT_DIR)
print("T_LIST", T_LIST)


In [ ]:
script = APP_ROOT / "inference" / (
    "run_ddim2ddim_inference.py" if MODE == "ddim2ddim" else "run_mixed_diffusion_inference.py"
)
cmd = [
    sys.executable, str(script),
    "--input-dir", str(INPUT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--model-path", str(MODEL_PATH),
    "--T", *[str(t) for t in T_LIST],
]
if MODE != "ddim2ddim":
    # mixed runner currently takes a single --T; use the first sweep value
    cmd = [
        sys.executable, str(script),
        "--input-dir", str(INPUT_DIR),
        "--output-dir", str(OUTPUT_DIR),
        "--model-path", str(MODEL_PATH),
        "--T", str(T_LIST[0]),
        "--mode", MODE,
    ]

print("cmd:\n ", " ".join(cmd))
# Uncomment to run:
# subprocess.run(cmd, check=True, cwd=str(APP_ROOT),
#                env={**dict(**__import__('os').environ),
#                     "PYTHONPATH": f"{APP_ROOT/'_stubs'}:{APP_ROOT/'train'/'diffusion-anomaly'}:{APP_ROOT/'inference'}"})
print("(dry-run)")
